# RAG Document Question-Answering System
### Custom document: *Space Exploration — Study Notes*

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline from
scratch and uses it to answer questions about a custom PDF document.

**Pipeline stages:**
1. Load the document and extract raw text
2. Split the text into overlapping chunks
3. Turn each chunk into a vector embedding
4. Store the vectors in a searchable index
5. Retrieve the chunks most relevant to a question
6. Generate an answer grounded in the retrieved chunks

The document used here is a set of study notes on **space exploration**
(solar system, satellites, Apollo program, rocket propulsion, space
telescopes, Mars exploration, the ISS, and private spaceflight).

## 1. Install & Import Dependencies

In [3]:
!pip install pypdf scikit-learn numpy -q --break-system-packages

import os
import re
import numpy as np
import pypdf

print("Dependencies loaded.")

Dependencies loaded.


## 2. Load the Document

`PDF_PATH` points at the custom document to index. Swap this for any
other `.pdf` file to run the same pipeline on different content.

In [4]:
PDF_PATH = "space_exploration_notes.pdf"

def extract_pdf_text(path: str) -> str:
    """Reads a PDF file and concatenates the text from every page."""
    reader = pypdf.PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)

document_text = extract_pdf_text(PDF_PATH)
print(f"Extracted {len(document_text)} characters from '{PDF_PATH}'.\n")
print(document_text[:400], "...")

Extracted 4045 characters from 'space_exploration_notes.pdf'.

Space Exploration: Study Notes
The Solar System
Our solar system consists of the Sun, eight planets, five recognized dwarf planets, and countless
smaller bodies such as asteroids, comets, and moons. The four inner planets — Mercury, Venus,
Earth, and Mars — are rocky and relatively small. The four outer planets — Jupiter, Saturn,
Uranus, and Neptune — are gas or ice giants with much larger diamete ...


## 3. Chunk the Text

Large documents need to be split into smaller passages so retrieval can
return focused, relevant context instead of the whole document. Chunks
overlap slightly so that ideas spanning a chunk boundary aren't lost.

In [5]:
def split_into_chunks(text: str, chunk_size: int = 110, overlap: int = 25) -> list:
    """Splits text into overlapping chunks of `chunk_size` words each."""
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")
    step = max(1, chunk_size - overlap)

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += step
    return chunks

document_chunks = split_into_chunks(document_text, chunk_size=110, overlap=25)
print(f"Document split into {len(document_chunks)} chunks.\n")
for i, c in enumerate(document_chunks):
    print(f"[{i}] {c[:90]}...")

Document split into 8 chunks.

[0] Space Exploration: Study Notes The Solar System Our solar system consists of the Sun, eigh...
[1] into orbit around a celestial body, most commonly Earth. Satellites are used for communica...
[2] the goal of landing humans on the Moon and returning them safely to Earth. Apollo 11, laun...
[3] expels propellant mass at high velocity out of a nozzle, and the reaction force propels th...
[4] Telescope, launched in 1990, has captured detailed images of distant galaxies and played a...
[5] rock samples to search for signs of past microbial life and study the planet's geology. Pe...
[6] from the United States, Russia, Europe, Japan, and Canada, and has been continuously occup...
[7] rocket boosters to land and be flown again rather than being discarded after a single use....


## 4. Embed the Chunks

Each chunk is converted into a numeric vector so that semantic
similarity can be measured with cosine similarity. This uses
`sentence-transformers` when it's available (real semantic embeddings),
and automatically falls back to TF-IDF otherwise — so the notebook still
runs in offline / restricted environments.

In [6]:
class ChunkEmbedder:
    """Embeds text chunks. Prefers sentence-transformers, falls back to TF-IDF."""

    def __init__(self):
        self.backend = None
        try:
            from sentence_transformers import SentenceTransformer
            self._model = SentenceTransformer("all-MiniLM-L6-v2")
            self.backend = "sentence-transformers"
        except Exception:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._vectorizer = TfidfVectorizer(stop_words="english")
            self.backend = "tfidf"
        print(f"[ChunkEmbedder] backend = {self.backend}")

    def fit_transform(self, texts: list) -> np.ndarray:
        """Fits (if needed) and encodes the full corpus."""
        if self.backend == "sentence-transformers":
            return np.array(self._model.encode(texts, show_progress_bar=False))
        matrix = self._vectorizer.fit_transform(texts)
        return matrix.toarray()

    def transform(self, texts: list) -> np.ndarray:
        """Encodes new text (e.g. a query) using the already-fitted embedder."""
        if self.backend == "sentence-transformers":
            return np.array(self._model.encode(texts, show_progress_bar=False))
        return self._vectorizer.transform(texts).toarray()


embedder = ChunkEmbedder()
chunk_vectors = embedder.fit_transform(document_chunks)
print(f"Embedded {chunk_vectors.shape[0]} chunks into vectors of size {chunk_vectors.shape[1]}.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[ChunkEmbedder] backend = sentence-transformers
Embedded 8 chunks into vectors of size 384.


## 5. Build a Vector Index

A minimal in-memory index that supports cosine-similarity search over
the chunk vectors. For larger document collections this could be
swapped for FAISS, Chroma, or another dedicated vector database without
changing the rest of the pipeline.

In [7]:
class SimpleVectorIndex:
    """Stores chunk vectors and finds the most similar ones to a query vector."""

    def __init__(self, chunks: list, vectors: np.ndarray):
        self.chunks = chunks
        self.vectors = vectors

    def most_similar(self, query_vector: np.ndarray, top_k: int = 3) -> list:
        query_norm = query_vector / (np.linalg.norm(query_vector) + 1e-10)
        doc_norms = self.vectors / (np.linalg.norm(self.vectors, axis=1, keepdims=True) + 1e-10)
        scores = doc_norms @ query_norm

        best_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best_idx]


vector_index = SimpleVectorIndex(document_chunks, chunk_vectors)
print("Vector index built.")

Vector index built.


## 6. Retrieve Relevant Chunks for a Question

Given a question, embed it with the same embedder and look up the
closest chunks in the index.

In [8]:
def retrieve_context(question: str, top_k: int = 3) -> list:
    query_vector = embedder.transform([question])[0]
    return vector_index.most_similar(query_vector, top_k=top_k)


sample_question = "What was the Apollo program?"
retrieved = retrieve_context(sample_question, top_k=3)

print(f"Question: {sample_question}\n")
for rank, (chunk, score) in enumerate(retrieved, 1):
    print(f"#{rank}  score={score:.3f}\n{chunk[:150]}...\n")

Question: What was the Apollo program?

#1  score=0.512
into orbit around a celestial body, most commonly Earth. Satellites are used for communication, weather monitoring, navigation systems such as GPS, an...

#2  score=0.492
the goal of landing humans on the Moon and returning them safely to Earth. Apollo 11, launched in July 1969, achieved this goal when astronauts Neil A...

#3  score=0.236
rock samples to search for signs of past microbial life and study the planet's geology. Perseverance, which landed in 2021, also carries the Ingenuity...



## 7. Generate an Answer

The generator turns retrieved chunks + the question into a final
answer. It tries a local `flan-t5` text-generation model if
`transformers` is available; otherwise it falls back to a lightweight
extractive method that returns the most relevant retrieved sentence(s)
directly — so the notebook produces a grounded answer either way.

In [9]:
class AnswerBuilder:
    """Builds an answer from a question + retrieved chunks."""

    def __init__(self):
        self.backend = None
        try:
            from transformers import pipeline
            self._pipe = pipeline("text2text-generation", model="google/flan-t5-base")
            self.backend = "flan-t5"
        except Exception:
            self.backend = "extractive"
        print(f"[AnswerBuilder] backend = {self.backend}")

    def answer(self, question: str, retrieved_chunks: list) -> str:
        context = "\n".join(chunk for chunk, _ in retrieved_chunks)

        if self.backend == "flan-t5":
            prompt = (
                f"Answer the question using only the context below.\n\n"
                f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
            )
            result = self._pipe(prompt, max_length=150, do_sample=False)
            return result[0]["generated_text"].strip()

        # Extractive fallback: return the top-scoring chunk, trimmed.
        top_chunk, top_score = retrieved_chunks[0]
        return (
            f"(extractive answer, no generative model available — "
            f"top match, score={top_score:.3f})\n{top_chunk}"
        )


answer_builder = AnswerBuilder()
print(answer_builder.answer(sample_question, retrieved))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

[AnswerBuilder] backend = extractive
(extractive answer, no generative model available — top match, score=0.512)
into orbit around a celestial body, most commonly Earth. Satellites are used for communication, weather monitoring, navigation systems such as GPS, and scientific research. Low Earth orbit, roughly 160 to 2,000 kilometers above the surface, is favored for imaging satellites because of its proximity to the ground, while geostationary orbit at about 35,786 kilometers allows a satellite to remain fixed above a single point on Earth. The Apollo Program The Apollo program was a series of NASA missions in the 1960s and early 1970s with the goal of landing humans on the Moon and returning them safely to Earth. Apollo 11, launched in July 1969, achieved this goal when astronauts


## 8. Full Pipeline — the `RAGSystem` Class

In [10]:
class RAGSystem:
    """End-to-end RAG pipeline: load -> chunk -> embed -> index -> retrieve -> generate."""

    def __init__(self, chunk_size: int = 110, overlap: int = 25, top_k: int = 3):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k
        self.embedder = ChunkEmbedder()
        self.answer_builder = AnswerBuilder()
        self.index = None

    def build_index(self, pdf_path: str):
        text = extract_pdf_text(pdf_path)
        self.chunks = split_into_chunks(text, self.chunk_size, self.overlap)
        vectors = self.embedder.fit_transform(self.chunks)
        self.index = SimpleVectorIndex(self.chunks, vectors)
        print(f"Indexed {len(self.chunks)} chunks from '{pdf_path}'.")

    def ask(self, question: str) -> dict:
        query_vector = self.embedder.transform([question])[0]
        retrieved = self.index.most_similar(query_vector, top_k=self.top_k)
        answer = self.answer_builder.answer(question, retrieved)
        return {"question": question, "answer": answer, "sources": retrieved}


rag = RAGSystem(chunk_size=110, overlap=25, top_k=3)
rag.build_index(PDF_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[ChunkEmbedder] backend = sentence-transformers
[AnswerBuilder] backend = extractive
Indexed 8 chunks from 'space_exploration_notes.pdf'.


## 9. Demo: Asking Questions About the Document

In [11]:
demo_questions = [
    "What is the difference between the inner and outer planets?",
    "How does a rocket generate thrust?",
    "What is the James Webb Space Telescope used for?",
    "Why has reusable rocket technology reduced launch costs?",
]

for q in demo_questions:
    result = rag.ask(q)
    print("Q:", result["question"])
    print("A:", result["answer"])
    top_chunk, top_score = result["sources"][0]
    print(f"   (top source, score={top_score:.3f}): {top_chunk[:120]}...")
    print("-" * 90)

Q: What is the difference between the inner and outer planets?
A: (extractive answer, no generative model available — top match, score=0.543)
Space Exploration: Study Notes The Solar System Our solar system consists of the Sun, eight planets, five recognized dwarf planets, and countless smaller bodies such as asteroids, comets, and moons. The four inner planets — Mercury, Venus, Earth, and Mars — are rocky and relatively small. The four outer planets — Jupiter, Saturn, Uranus, and Neptune — are gas or ice giants with much larger diameters and are surrounded by ring systems and numerous moons. Artificial Satellites An artificial satellite is a human-made object placed into orbit around a celestial body, most commonly Earth. Satellites are used for communication, weather monitoring, navigation systems such as GPS, and scientific research. Low
   (top source, score=0.543): Space Exploration: Study Notes The Solar System Our solar system consists of the Sun, eight planets, five recognized 

## 10. Try Your Own Question

In [12]:
your_question = "What does the ISS study about the human body?"

result = rag.ask(your_question)
print("Q:", result["question"])
print("A:", result["answer"])
print("\nRetrieved context used to answer:")
for i, (chunk, score) in enumerate(result["sources"], 1):
    print(f"\n[{i}] score={score:.3f}\n{chunk}")

Q: What does the ISS study about the human body?
A: (extractive answer, no generative model available — top match, score=0.466)
rock samples to search for signs of past microbial life and study the planet's geology. Perseverance, which landed in 2021, also carries the Ingenuity helicopter, the first aircraft to achieve powered, controlled flight on another planet, and is caching rock samples for a future mission intended to return them to Earth. The International Space Station The International Space Station, or ISS, is a habitable artificial satellite in low Earth orbit that serves as a microgravity research laboratory. It is a collaborative project involving space agencies from the United States, Russia, Europe, Japan, and Canada, and has been continuously occupied by rotating crews of astronauts since November 2000. Experiments aboard the

Retrieved context used to answer:

[1] score=0.466
rock samples to search for signs of past microbial life and study the planet's geology. Persev

## Conclusion

This notebook implements a complete RAG pipeline — **load → chunk →
embed → index → retrieve → generate** — and applies it to a custom PDF
of space exploration study notes. It uses stronger models
(`sentence-transformers`, `flan-t5`) automatically when they're
available, and degrades gracefully to lightweight alternatives (TF-IDF,
NumPy cosine similarity, extractive answers) when they aren't, so it
runs the same way in a restricted sandbox, in Google Colab, or on a
local machine.

**Possible extensions:**
- Swap `SimpleVectorIndex` for FAISS or Chroma for larger document sets
- Add a cross-encoder re-ranking step after initial retrieval
- Support multi-document collections instead of a single PDF
- Log retrieval scores over a test set of Q&A pairs to evaluate accuracy